In [2]:
import tensorflow as tf
from PIL import Image
import numpy as np
import keras
import os
from keras.applications import MobileNetV2

2026-11-07 10:32:40.382610: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [1]:
data_dir = "/Users/sajadeen/Desktop/GitHub/waste-reducing-app-hackathon/images/"
test_image = "/Users/sajadeen/Desktop/GitHub/waste-reducing-app-hackathon/test.png"

In [4]:
converted_count = 0
deleted_count = 0

for dir in os.listdir(data_dir):
    if dir == ".DS_Store":
        continue
    for file in os.listdir(os.path.join(data_dir, dir)):
        if file == ".DS_Store":
            continue
        allowed_exts = ["jpeg", "png", "gif", "bmp"]
        try:
            with Image.open(os.path.join(data_dir, dir, file)) as image_loaded:
                image_loaded.verify()

            with Image.open(os.path.join(data_dir, dir, file)) as image_loaded:
                acutal_ext = image_loaded.format.lower()
                if acutal_ext not in allowed_exts:
                    image_loaded.save(
                        os.path.join(data_dir, dir, file.split(".")[0] + ".png"),
                        "PNG",
                        icc_profile=None,
                    )
                    os.remove(os.path.join(data_dir, dir, file))
                    converted_count += 1
                    print(f"Converted {file} to PNG")
                    file = os.path.join(data_dir, dir, file.split(".")[0] + ".png")
                    if image_loaded.mode in ("RGBA", "P", "CMYK"):
                        image_loaded = image_loaded.convert("RGB")
                else:
                    image_loaded.save(
                        os.path.join(data_dir, dir, file),
                        image_loaded.format,
                        icc_profile=None,
                    )
        except Exception as e:
            os.remove(os.path.join(data_dir, dir, file))
            deleted_count += 1
            print(f"Deleted {file} due to error")

print(f"Converted: {converted_count}")
print(f"Deleted: {deleted_count}")

Converted depositphotos_40630785-stock-photo-pile-of-broken-eggshells.jpg to PNG
Converted -shells-with-warm-water-used-as-a-calcium-supplement-and-popular-in-asia-and.jpg to PNG
Converted depositphotos_42000011-stock-photo-broken-chicken-eggshells.jpg to PNG
Converted crushing-eggshells.jpg.webp to PNG
Converted 30322484-clay-pottery-ceramics-in-background-broken-glass-debris-design-wall.jpg to PNG
Converted kintsugi-art-japanese-broken-pottery.jpeg to PNG
Converted How-to-dispose-of-broken-glass-Cover.webp to PNG
Converted de-bowl-in-the-shape-of-a-star-at-the-bottom-of-the-bowl-melted-broken-glass.jpg to PNG
Converted Broken-Ceramics-Glass-Mirror-Chips.webp to PNG
Converted depositphotos_234468602-stock-photo-collage-tile-decoration-broken-glass.jpg to PNG
Converted tion-broken-glass-mosaic-broken-ceramic-tiles-as-crazy-background-for-paving.jpg to PNG
Converted why-does-ceramic-break-glass.jpg to PNG
Converted Is-Recycling-Glass-Worth-It.jpg to PNG
Converted depositphotos_66088493-

/Users/sajadeen/Desktop/GitHub/waste-reducing-app-hackathon/.venv/lib/python3.12/site-packages/PIL/TiffImagePlugin.py:982: UserWarning: Truncated File Read
  warnings.warn(str(msg))


Converted 474911884-H.webp to PNG
Converted 74806254.jpg to PNG
Converted pile-discarded-used-baby-diapers-footage-170531202_prevstill.jpeg to PNG
Converted: 80
Deleted: 3


In [6]:
data_augmentation = keras.Sequential(
    [
        keras.layers.RandomFlip("horizontal"),
        keras.layers.RandomRotation(0.1),
        keras.layers.RandomZoom(0.2),
        keras.layers.RandomFlip("vertical"),
        keras.layers.RandomContrast(0.2),
        keras.layers.RandomTranslation(0.1, 0.1)
    ]
)

In [7]:
train_ds = keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="training",
    image_size=(224, 224),
    batch_size=32,
    seed=42,
)

val_ds = keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="validation",
    image_size=(224, 224),
    batch_size=32,
    seed=42,
)

class_names = train_ds.class_names
num_classes = len(class_names)

Found 2791 files belonging to 17 classes.
Using 2233 files for training.
Found 2791 files belonging to 17 classes.
Using 558 files for validation.


In [8]:
base_model = MobileNetV2(
    input_shape=(224, 224, 3), include_top=False, weights="imagenet"
)

base_model.trainable = False

In [9]:
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)

In [10]:
model = keras.Sequential(
    [
        data_augmentation,
        keras.layers.Lambda( 
            lambda x: keras.applications.mobilenet_v2.preprocess_input(x)
        ),
        base_model,
        keras.layers.GlobalAveragePooling2D(),
        keras.layers.Dropout(0.2),
        keras.layers.Dense(num_classes, activation="softmax"),
    ]
)

In [11]:
model.compile(
    loss="sparse_categorical_crossentropy", optimizer="adam", metrics=["accuracy"]
)

model.fit(train_ds, validation_data=val_ds, epochs=15)

Epoch 1/15


/Users/sajadeen/Desktop/GitHub/waste-reducing-app-hackathon/.venv/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


70/70 ━━━━━━━━━━━━━━━━━━━━ 101s 1s/step - accuracy: 0.2503 - loss: 2.5738 - val_accuracy: 0.4749 - val_loss: 1.7942
Epoch 2/15
70/70 ━━━━━━━━━━━━━━━━━━━━ 95s 1s/step - accuracy: 0.4877 - loss: 1.6848 - val_accuracy: 0.5591 - val_loss: 1.5144
Epoch 3/15
70/70 ━━━━━━━━━━━━━━━━━━━━ 96s 1s/step - accuracy: 0.5602 - loss: 1.4408 - val_accuracy: 0.5645 - val_loss: 1.4119
Epoch 4/15
70/70 ━━━━━━━━━━━━━━━━━━━━ 93s 1s/step - accuracy: 0.5876 - loss: 1.3061 - val_accuracy: 0.6201 - val_loss: 1.3142
Epoch 5/15
70/70 ━━━━━━━━━━━━━━━━━━━━ 93s 1s/step - accuracy: 0.6391 - loss: 1.2042 - val_accuracy: 0.6165 - val_loss: 1.3086
Epoch 6/15
70/70 ━━━━━━━━━━━━━━━━━━━━ 94s 1s/step - accuracy: 0.6507 - loss: 1.1281 - val_accuracy: 0.6416 - val_loss: 1.2980
Epoch 7/15
70/70 ━━━━━━━━━━━━━━━━━━━━ 93s 1s/step - accuracy: 0.6597 - loss: 1.1135 - val_accuracy: 0.6308 - val_loss: 1.2827
Epoch 8/15
70/70 ━━━━━━━━━━━━━━━━━━━━ 70s 995ms/step - accuracy: 0.6708 - loss: 1.0512 - val_accuracy: 0.6290 - val_loss: 1.2793

In [12]:
base_model.trainable = True

for layer in base_model.layers[:100]:
    layer.trainable = False

In [13]:
custom_omtimizer = keras.optimizers.Adam(learning_rate=1e-4)

model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer=custom_omtimizer,
    metrics=["accuracy"]
)

In [ ]:
early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=5, restore_best_weights=True
)
model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=40,
    initial_epoch=15,
)

model.save("waste_classification_model.keras")

Epoch 16/40
 6/70 ━━━━━━━━━━━━━━━━━━━━ 1:01 966ms/step - accuracy: 0.7708 - loss: 0.7762